# Case Study: Feed Recommendation System

This is the most common ML system design interview problem. The following worked answer follows the 7-step framework from mlsd0, written in the interview transcript format with interviewer commentary.

## What Interviewers Test
- Can you scope a recommendation problem with real numbers?
- Do you define metrics before choosing architecture?
- Can you explain the two-tower + DNN ranker pattern?
- Do you address cold-start, diversity, and exploration?
- Can you reason through serving latency at scale?

## Transcript: Design a Video/News Feed Recommender

**Interviewer:** Design the recommendation system for a video-sharing platform. You have 500M daily active users and 100M videos.

---

**You:** Let me clarify a few things first.

1. *Goal:* Are we optimizing for engagement (watch-time) or for something else (subscription conversion, satisfaction)?
2. *Latency:* What's the p99 SLA for feed loading? I'll assume 200ms.
3. *Freshness:* Do we need to incorporate the last few minutes of behavior, or daily-level signals?
4. *Geography:* Single region or global? I'll assume global, multi-datacenter.

**Interviewer:** Engagement primarily. 200ms p99. Daily signals are fine to start.

---

**Step 1 — Scale Numbers:**
```
500M DAU × 3 sessions/day × 5 feed loads = 7.5B requests/day ≈ 87K QPS
100M videos × average 100 features = 10B feature values to index
```

**Step 2 — Metrics:**
- North-star: watch-time per session (long-term)
- Proxy offline: NDCG@10 on watch-time signal
- Guardrails: not reduce CTR; not increase clickbait rate; p99 latency < 200ms

> 💡 **Interview Tip:** Giving real numbers (87K QPS, 10B feature values) signals that you've built at scale, not just read papers.


## Data, Features, Model

**Step 3 — Labels:**
- Primary: watch-time ratio (watched/video_duration) as continuous label
- Secondary: explicit signals (like, share, skip < 5s = negative)
- Collect from logs; IPW-correct for position bias

**Step 4 — Features:**

| Feature family | Examples | Freshness |
|---|---|---|
| User long-term | Watch history embeddings, subscriptions | Daily |
| User session | Last 3 videos watched this session | Real-time |
| Video | Content embedding, category, upload time | Daily |
| Video stats | 7d watch-rate, trending score | Hourly |
| Context | Time of day, device type, language | Per-request |
| Cross | User × video category affinity | Daily |

**Step 5 — Model (baseline → v2):**

1. **Baseline:** Popularity ranker (top trending videos globally). Establishes the floor. Can be precomputed.
2. **v1:** Collaborative filtering (matrix factorization). Offline AUC improvement, handles personalization.
3. **v2:** Two-tower retrieval + DNN ranker. Two-tower retrieves top-500 candidates via ANN; a separate DNN ranker scores them with full features.


In [ ]:
# Illustrative: two-tower scoring + ranking simulation
import numpy as np
np.random.seed(42)

n_users, n_videos, embed_dim = 100, 50000, 64

# Pre-computed item embeddings (in production: stored in FAISS)
video_embeddings = np.random.randn(n_videos, embed_dim)
video_embeddings /= np.linalg.norm(video_embeddings, axis=1, keepdims=True)

def get_user_embedding(user_id):
    """In production: query tower forward pass on user features."""
    np.random.seed(user_id)
    emb = np.random.randn(embed_dim)
    return emb / np.linalg.norm(emb)

def retrieve_candidates(user_emb, video_embeddings, k=500):
    """Brute-force ANN (in production: FAISS HNSW)."""
    scores = video_embeddings @ user_emb
    return np.argpartition(-scores, k)[:k]

def rank_candidates(candidates, user_id, n_final=20):
    """Simulate DNN ranker with richer features."""
    # In production: fetch full features and run DNN ranker
    np.random.seed(user_id * 100)
    ranker_scores = np.random.rand(len(candidates))  # placeholder
    top_idx = np.argpartition(-ranker_scores, n_final)[:n_final]
    return candidates[top_idx], ranker_scores[top_idx]

# Simulate serving for one user
user_emb = get_user_embedding(user_id=42)
candidates = retrieve_candidates(user_emb, video_embeddings, k=500)
final_videos, final_scores = rank_candidates(candidates, user_id=42, n_final=20)

print(f"Retrieved {len(candidates)} candidates via ANN")
print(f"Ranked to {len(final_videos)} final results")
print(f"Top 5 video IDs: {final_videos[:5]}")


## Serving Architecture & Monitoring

**Step 6 — Serving funnel:**
```
Request (user_id, context)
  ↓  [User embedding lookup + real-time features]  <5ms
  ↓  [ANN retrieval: 100M → 500 candidates]        <10ms
  ↓  [Feature assembly (feature store)]             <15ms
  ↓  [DNN ranker: 500 → 20 results]                <30ms
  ↓  [Diversity filter + dedup]                     <5ms
Response  (total <65ms — well within 200ms SLA)
```

**Step 7 — Monitoring:**
- Watch-time per session (north-star) — daily A/B tracking
- PSI on input features — hourly
- Score distribution — per-request sampling
- Diversity metric (pct long-tail videos) — anti-popularity-bias guard

**Cold-start handling:**
- New users: serve globally trending + regionally popular content; use signup preferences
- New videos: retrieve based on content embeddings (title/thumbnail); boost freshness score

**Exploration:** Reserve 5% of slots for Thompson Sampling to surface new/niche content.


## Where Modern Recsys Has Moved

The two-tower + ranker design above is still the correct default answer, and still what most
production systems run. But a strong interviewer will push past it, and these three directions
are where they push.

### 1. Sequential recommendation
Two-tower models compress a user into a *static* embedding built from aggregated history. That
throws away order — and order carries most of the intent signal. Someone who just watched three
pasta recipes is in a different state than their long-run average suggests.

Sequential recommenders (the SASRec/BERT4Rec lineage) instead run a transformer over the user's
recent interaction sequence and predict the next item directly. The user representation becomes
a *function of the session*, refreshed per request rather than per day.

**Cost:** you can no longer precompute the user embedding, which changes the serving budget.
Common compromise: a cached long-term embedding concatenated with a short sequence encoder over
the last N interactions computed at request time.

### 2. Generative retrieval and semantic IDs
Standard retrieval is: embed the query, run ANN search over an item index. Generative retrieval
replaces that with: **generate the item identifier directly.**

Each item is assigned a *semantic ID* — a short sequence of discrete tokens from hierarchical
quantization of its content embedding, so similar items share prefixes. The model then
autoregressively generates the ID of the item to recommend.

| | ANN retrieval | Generative retrieval |
|---|---|---|
| Item index | Explicit vector index (FAISS/HNSW) | Implicit in model weights |
| New items | Add to index immediately | Needs ID assignment; may need retraining |
| Scaling | Index grows with catalog | Model size roughly constant |
| Maturity | Battle-tested everywhere | Newer; strong published results, deployment still less common |

### 3. LLM-augmented ranking
Three practical uses, in increasing order of ambition:
- **Feature generation (most common, lowest risk).** Use an LLM offline to produce rich item
  descriptions, topic labels, and quality scores that feed the existing ranker. No latency cost,
  because it's a batch job.
- **Cold-start via semantics.** A brand-new item with zero engagement still has a title,
  description, and thumbnail. LLM embeddings give it a usable representation on day one.
- **Direct LLM ranking (rare online).** Prompt an LLM to rank candidates. Quality can be strong,
  but latency and cost make it impractical in a 200ms funnel today — it appears mostly offline,
  or as a teacher whose judgements are distilled into the production ranker.

> 💡 **Interview Tip:** Do not open with any of this. Open with the funnel above — it shows you
> know what actually ships. Bring these up as the *v3 roadmap* when asked "what would you do
> next," and pair each with its cost. Leading with generative retrieval on a 500M-DAU system
> reads as someone who has read papers rather than shipped systems.

## 8 Follow-up Questions & Strong Answers

**Q: How do you handle a video that goes viral in the last 10 minutes?**
The daily-refreshed embeddings won't capture it. Add a "trending score" feature computed in near-real-time from a Flink pipeline watching engagement velocity. This feature is available in the online feature store within minutes. Also add a freshness bonus to the ranker.

**Q: How do you avoid the popularity bias spiral?**
Reserve an exploration budget (5–10% of slots) for non-popular items. Log counterfactual scores. Train the ranker with IPW to debias observed engagement. Monitor diversity metrics on the served distribution — if the Gini coefficient of video popularity increases, tighten the exploration budget.

**Q: How would you add a "not interested" feedback loop?**
Collect explicit negative signals (skip, "not interested" button). Add them as negative labels in training. In the ranker, penalize videos from categories with high skip rates for that user. Monitor that adding negatives doesn't reduce engagement by more than the guardrail threshold.

**Q: What changes if we need sub-50ms instead of 200ms?**
Use pre-computed user embeddings (refresh every few minutes, not per-request). Reduce model complexity in the ranker (fewer layers, quantize to int8). Cache ranked results per (user_id, time_bucket) for similar users. Use gRPC instead of REST for the model server.

**Q: How do you evaluate if the system is getting better over time?**
Holdout experiment: keep 1% of users on the old model indefinitely. Measure cumulative watch-time and satisfaction over weeks. Short-term A/B tests are susceptible to novelty effects; the holdout captures long-term behavior change.

**Q: How would you handle multiple countries with different content preferences?**
Train country-specific models or add country as a feature. Use transfer learning: pretrain globally, fine-tune per region. Ensure training data includes regional signals and that the feature store has local trending data. Be careful about regulatory requirements (content restrictions by country).

**Q: What's the biggest risk when you retrain daily?**
Training data distribution shift can cause sudden metric drops. Mitigate: monitor evaluation metrics on a fixed holdout before promoting; use canary rollouts; maintain warm-start from yesterday's weights; add a human-review gate for large metric swings.

**Q: How do you handle a major event (e.g., breaking news) that shifts all user interests?**
The daily-retrained model will be stale for the event's first hours. Mitigate with: real-time trending features (computed from last-hour signals), editorial curation for breaking news, and a manual override mechanism to inject relevant content into the top of the feed.

## Key Takeaways
- Recsys = candidate retrieval (millions → 500) + ranking (500 → 20) + re-ranking (diversity)
- Two-tower for retrieval; DNN ranker for final ranking with full features
- Cold-start: content embeddings for new items; regional trending for new users
- Exploration budget (5–10%) prevents popularity bias spiral
- Freshness features from real-time pipeline catch viral content that daily-retrained models miss
- Monitor watch-time, feature PSI, score distribution, and diversity